# Manual CHECKER for single after feat. extraction
lisa - Gustavo Cerati

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

def sanitize_channel_name(ch_name):
    return str(ch_name).replace(" ", "_").replace("-", "_")

def check_one_window_features(df_features, row_idx, npz_base_path, atol=1e-10):
    row = df_features.iloc[row_idx]

    file_name = row["file_name"]
    start_sample = int(row["start_sample"])
    end_sample = int(row["end_sample"])

    npz_path = Path(npz_base_path) / file_name
    data = np.load(npz_path, allow_pickle=True)

    X = data["X"]
    channel_names = data["channel_names"]

    window = X[:, start_sample:end_sample]

    print("File:", file_name)
    print("Window ID:", row["window_id"])
    print("Window shape:", window.shape)
    print("Start sample:", start_sample)
    print("End sample:", end_sample)

    for ch in range(window.shape[0]):
        ch_name = sanitize_channel_name(channel_names[ch])
        x = window[ch, :]

        manual_mean = np.mean(x)
        manual_std = np.std(x)
        manual_var = np.var(x)
        manual_rms = np.sqrt(np.mean(x**2))
        manual_ptp = np.ptp(x)
        manual_line_length = np.sum(np.abs(np.diff(x)))

        checks = {
            f"mean_{ch_name}": manual_mean,
            f"std_{ch_name}": manual_std,
            f"var_{ch_name}": manual_var,
            f"rms_{ch_name}": manual_rms,
            f"ptp_{ch_name}": manual_ptp,
            f"line_length_{ch_name}": manual_line_length,
        }

        print(f"\nChannel {ch}: {ch_name}")

        for col, manual_value in checks.items():
            pipeline_value = row[col]
            diff = abs(pipeline_value - manual_value)

            print(
                f"{col}: "
                f"pipeline={pipeline_value:.6f}, "
                f"manual={manual_value:.6f}, "
                f"diff={diff:.2e}, "
                f"OK={diff < atol}"
            )

In [7]:
df_feat_ictalVspreictal_1min = pd.read_pickle("/home/tperezsanchez/FoundationModel_EEG_Dissertation/Main_project/results/JYXFE/Feature_ext/Part2_features/JYXFE_IN-normalized_npz_FP-fullnpz_W10s_PRE6to5min_ICT0to1min_GAPasINT_FINAL-PREvsSEIZ_20260511_v01_FEAT-TIME-FREQ_20260511_v01/JYXFE_IN-normalized_npz_FP-fullnpz_W10s_PRE6to5min_ICT0to1min_GAPasINT_FINAL-PREvsSEIZ_20260511_v01_FEAT-TIME-FREQ_20260511_v01_df_features_ictalVspreictal.pkl")
npz_base_path= "/home/tperezsanchez/FoundationModel_EEG_Dissertation/Main_project/results/JYXFE/Pre_processing/JYXFE_IN-npz_GLOBALCH-NORM_CH2_EPS1e-08_20260511/normalized_npz"
check_one_window_features(
    df_features=df_feat_ictalVspreictal_1min,
    row_idx=0,
    npz_base_path=npz_base_path
)

File: JYXFE_22_preproc_full.npz
Window ID: 983
Window shape: (2, 2070)
Start sample: 2034810
End sample: 2036880

Channel 0: EEG_SQ_D_SQ_C
mean_EEG_SQ_D_SQ_C: pipeline=0.004530, manual=0.004530, diff=0.00e+00, OK=True
std_EEG_SQ_D_SQ_C: pipeline=0.648885, manual=0.648885, diff=0.00e+00, OK=True
var_EEG_SQ_D_SQ_C: pipeline=0.421052, manual=0.421052, diff=0.00e+00, OK=True
rms_EEG_SQ_D_SQ_C: pipeline=0.648901, manual=0.648901, diff=0.00e+00, OK=True
ptp_EEG_SQ_D_SQ_C: pipeline=4.279262, manual=4.279262, diff=0.00e+00, OK=True
line_length_EEG_SQ_D_SQ_C: pipeline=504.982086, manual=504.982086, diff=0.00e+00, OK=True

Channel 1: EEG_SQ_P_SQ_C
mean_EEG_SQ_P_SQ_C: pipeline=-0.005767, manual=-0.005767, diff=0.00e+00, OK=True
std_EEG_SQ_P_SQ_C: pipeline=0.674758, manual=0.674758, diff=0.00e+00, OK=True
var_EEG_SQ_P_SQ_C: pipeline=0.455299, manual=0.455299, diff=0.00e+00, OK=True
rms_EEG_SQ_P_SQ_C: pipeline=0.674783, manual=0.674783, diff=0.00e+00, OK=True
ptp_EEG_SQ_P_SQ_C: pipeline=9.588884, m

1. The correct window is being extracted.
2. The start/end samples are correctly matched.
3. The channels are not being mixed.
4. The columns are correctly assigned to the corresponding channel.
5. The features are being correctly stored in the dataframe.
